# JSON Structured Output

По умолчанию LLM возвращает свободный текст — строку, которую нужно парсить вручную.  
**Structured Output** заставляет модель вернуть строго типизированный объект по заданной схеме.

## Как это работает

```
Prompt → LLM → function call → JSON → Pydantic объект
```

1. Мы определяем **Pydantic-схему** — она автоматически превращается в JSON Schema.
2. LangChain передаёт схему в LLM как инструмент (function calling / tool use).
3. LLM «вызывает» этот инструмент с заполненными полями.
4. LangChain десериализует результат обратно в Python-объект.

> Groq, OpenAI и другие провайдеры поддерживают это через стандартный механизм function calling.

In [ ]:
%pip install -q langchain langchain-groq pydantic python-dotenv

## Настройка

In [ ]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()  # читает GROQ_API_KEY из .env

llm = ChatGroq(model="llama3-8b-8192", temperature=0)

## 1. Определяем схему

Схема — обычная Pydantic-модель. Поля с `Field(description=...)` помогают LLM понять,  
что именно нужно вернуть в каждом поле.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
import json

class ReviewAnalysis(BaseModel):
    """Анализ отзыва пользователя."""

    sentiment: Literal["позитивный", "нейтральный", "негативный"] = Field(
        description="Общий тон отзыва"
    )
    confidence: float = Field(
        ge=0.0, le=1.0,
        description="Уверенность классификатора от 0 до 1"
    )
    key_topics: list[str] = Field(
        description="Главные темы, упомянутые в отзыве (2–4 штуки)"
    )
    action_required: bool = Field(
        description="Требует ли отзыв ответа от команды поддержки"
    )
    summary: str = Field(
        description="Краткое резюме отзыва в одном предложении"
    )


# Посмотрим на JSON Schema, которую увидит LLM
print(json.dumps(ReviewAnalysis.model_json_schema(), indent=2, ensure_ascii=False))


## 2. Оборачиваем LLM

`with_structured_output()` — единственное изменение по сравнению с обычным вызовом.

In [ ]:
structured_llm = llm.with_structured_output(ReviewAnalysis)

# Тип возвращаемого значения — ReviewAnalysis, а не str
print(type(structured_llm))

## 3. Используем

In [ ]:
review = """
Приложение работает быстро и интерфейс интуитивный, но вчера потерял все данные после обновления.
Никакого предупреждения не было. Поддержка не ответила уже 3 дня. Очень расстроен.
"""

result: ReviewAnalysis = structured_llm.invoke(
    f"Проанализируй следующий отзыв пользователя:\n{review}"
)

# Работаем с объектом, а не парсим строку
print(f"Тональность:      {result.sentiment}")
print(f"Уверенность:      {result.confidence:.0%}")
print(f"Темы:             {', '.join(result.key_topics)}")
print(f"Нужен ответ:      {'да' if result.action_required else 'нет'}")
print(f"Резюме:           {result.summary}")

## 4. Сериализация в JSON

Pydantic-объект сразу готов к отдаче через API — без дополнительного кода.

In [ ]:
print(result.model_dump_json(indent=2))

## 5. Пакетная обработка

Structured output хорошо масштабируется — каждый ответ гарантированно одной структуры.

In [ ]:
reviews = [
    "Отличный продукт, пользуюсь каждый день!",
    "Ничего особенного, обычное приложение",
    "Полный провал, удалил после первого использования. Требую возврат денег!",
]

for i, text in enumerate(reviews, 1):
    r: ReviewAnalysis = structured_llm.invoke(f"Проанализируй отзыв: {text}")
    flag = "⚠️ " if r.action_required else "   "
    print(f"{flag}[{r.sentiment:12}] {r.confidence:.0%}  {text[:50]}")